# TMR AI Text Detector — Colab API Server
**Model:** `Oxidane/tmr-ai-text-detector` (RoBERTa trained on RAID dataset)

### Steps:
1. Run **Cell 1** (install deps) — takes ~1 min
2. Run **Cell 2** (start server) — wait for the public URL to appear
3. Copy the `Running on public URL: https://xxxx.gradio.live` link
4. Paste it into your ATS Agent `.env` as `COLAB_DETECTOR_URL=https://xxxx.gradio.live`
5. Restart your Flask server

> **Keep this Colab tab open** while using the AI Lab — closing it stops the server.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers torch gradio

In [ ]:
# Cell 2 — Load model and launch public API server
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Loading model Oxidane/tmr-ai-text-detector ...")
MODEL_PATH = "Oxidane/tmr-ai-text-detector"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"[OK] Model loaded on {device.upper()}")

def detect(text: str):
    """Return AI probability for the input text."""
    if not text or len(text.strip()) < 20:
        return {"error": "Text too short"}
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)[0]
    ai_prob    = round(probs[1].item() * 100, 2)
    human_prob = round(probs[0].item() * 100, 2)
    verdict    = "AI" if ai_prob >= 50 else "Human"
    label      = "AI-Generated" if verdict == "AI" else "Likely Human"
    return {
        "ai_probability":    ai_prob,
        "human_probability": human_prob,
        "verdict":  verdict,
        "label":    label,
    }

# Gradio interface — gives a free public URL automatically
with gr.Blocks(title="TMR AI Text Detector API") as demo:
    gr.Markdown("## TMR AI Text Detector\nPowered by `Oxidane/tmr-ai-text-detector` (RAID-trained RoBERTa)")
    with gr.Row():
        txt = gr.Textbox(label="Input Text", lines=6, placeholder="Paste text to analyze...")
        out = gr.JSON(label="Result")
    btn = gr.Button("Analyze", variant="primary")
    btn.click(detect, inputs=txt, outputs=out)
    # Also expose as a raw API endpoint at /run/predict
    gr.Interface(fn=detect, inputs="text", outputs="json", allow_flagging="never").render()

print("\n" + "="*60)
print("  Launching public API server...")
print("  Copy the 'Running on public URL' link below")
print("  and paste it into your .env as COLAB_DETECTOR_URL=")
print("="*60 + "\n")

demo.launch(share=True, show_error=True)